# Métricas de avaliação — classificação

**Objetivo:** num conjunto **desbalanceado**, mostrar como a acurácia engana, calcular a matriz de confusão e o `classification_report`, e desenhar a curva ROC.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paleta do curso (idêntica ao site)
INK, PAPER = "#1b1c12", "#f3f6e0"
BLUE, RED, GREEN, MUTED = "#3266ad", "#c0392b", "#1a7a4a", "#66693f"

plt.rcParams.update({
    "font.family": "serif", "font.size": 12,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER,
    "axes.edgecolor": "#adb87f", "axes.grid": True,
    "grid.color": "#d7deb2", "grid.linewidth": 0.7,
    "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
})
rng = np.random.default_rng(42)

## Dados desbalanceados (5% de positivos)

In [ ]:
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=2000, weights=[0.95, 0.05],
                           n_informative=5, random_state=42)
print('proporção de positivos:', round(y.mean(), 3))

## O classificador trivial 'tudo negativo'

In [ ]:
acc_trivial = (y == 0).mean()
print(f'Acurácia prevendo sempre 0: {acc_trivial:.3f}  <- alta e inútil')

## Um modelo de verdade

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3,
                                      stratify=y, random_state=42)
clf = LogisticRegression(max_iter=500).fit(Xtr, ytr)
pred = clf.predict(Xte)
print(confusion_matrix(yte, pred))
print(classification_report(yte, pred, digits=3))

## Curva ROC e AUC

In [ ]:
from sklearn.metrics import RocCurveDisplay

proba = clf.predict_proba(Xte)[:, 1]
print('AUC:', round(roc_auc_score(yte, proba), 3))
RocCurveDisplay.from_predictions(yte, proba, name='LogReg')
plt.plot([0, 1], [0, 1], '--', color=MUTED)
plt.title('Curva ROC'); plt.show()

## Exercícios

**1.** Baixe o limiar de decisão para 0,2 (em vez de 0,5). O que acontece com recall e precisão da classe positiva?

**2.** Por que a AUC não muda ao alterar o limiar, mas a acurácia muda?

In [ ]:
# @title Solução (clique para revelar)
from sklearn.metrics import precision_score, recall_score
for thr in [0.5, 0.2]:
    p = (proba >= thr).astype(int)
    print(f'limiar={thr}: recall={recall_score(yte, p):.3f} '
          f'precisão={precision_score(yte, p):.3f}')
# A AUC integra TODOS os limiares, então não depende de um limiar
# específico; acurácia, precisão e recall são medidas em um limiar.